# Imports 

In [1]:
import pandas as pd
import os
import re
import csv
import seaborn as sns
import matplotlib.pyplot as plt
import zipfile
import numpy as np
import matplotlib.patches as mpatches

# Dataset: import and column selection

### Read files

In [2]:
output_dir = r"../inputs/FED_data"

In [3]:
### Read all unzipped files stored locally
dfs = {}

for root, dirs, files in os.walk(output_dir):
    for dir_ in dirs[::-1]:
        for file in os.listdir(os.path.join(output_dir, dir_)):
            if file.startswith("BHCF") and file.endswith(".txt"):
                # extract date to add to col
                date_str = file.replace("BHCF", "").replace(".txt", "")

                # code to read file
                file_path = os.path.join(output_dir, dir_, file)
                print(f"file_path: {file_path.split('/')[-1]}")
                print("--")
                try:
                    df = pd.read_csv(file_path, sep="^", encoding="latin1", dtype=str)
                except:
                    df = pd.read_csv(
                        file_path,
                        sep="^",
                        encoding="latin1",
                        dtype=str,
                        quoting=csv.QUOTE_NONE,
                    )

                # Add quarter column
                df["quarter"] = date_str

                # Store dataframe in dictionary with key = date string
                dfs[date_str] = df
                break
            break
        # break

file_path: FED_data\BHCF20250630\BHCF20250630.txt
--
file_path: FED_data\BHCF20250331\BHCF20250331.txt
--
file_path: FED_data\BHCF20241231\BHCF20241231.txt
--
file_path: FED_data\BHCF20240930\BHCF20240930.txt
--
file_path: FED_data\BHCF20240630\BHCF20240630.txt
--
file_path: FED_data\BHCF20240331\BHCF20240331.txt
--
file_path: FED_data\BHCF20231231\BHCF20231231.txt
--
file_path: FED_data\BHCF20230930\BHCF20230930.txt
--
file_path: FED_data\BHCF20230630\BHCF20230630.txt
--
file_path: FED_data\BHCF20230331\BHCF20230331.txt
--
file_path: FED_data\BHCF20221231\BHCF20221231.txt
--
file_path: FED_data\BHCF20220930\BHCF20220930.txt
--
file_path: FED_data\BHCF20220630\BHCF20220630.txt
--
file_path: FED_data\BHCF20220331\BHCF20220331.txt
--
file_path: FED_data\BHCF20211231\BHCF20211231.txt
--
file_path: FED_data\BHCF20210930\BHCF20210930.txt
--
file_path: FED_data\BHCF20210630\BHCF20210630.txt
--
file_path: FED_data\BHCF20210331\BHCF20210331.txt
--
file_path: FED_data\BHCF20201231\BHCF20201231.

### Upload selected columns data

Analysed each individual column from the database. Mid exercise I found one of the reports that returns data to National Information Center with the code to each line of the P&L.

- FR Y -9c 
    - financial statement: https://www.federalreserve.gov/reportforms/forms/FR_Y-9C20180930_f.pdf
    - detail on financial statement rubrics: https://www.federalreserve.gov/apps/reportingforms/Download/DownloadAttachment?guid=d036ea09-75d3-4f2f-8fe3-0f43c76d2f70

- FR_Y-9LP (Parent Company Only Financial Statements for Large Bank Holding Companies): 
    - financial statement: https://www.federalreserve.gov/apps/reportingforms/Download/DownloadAttachment?guid=e56cd829-8369-4263-96ca-4de4e12ce585#:~:text=03/2024-,For%20Federal%20Reserve%20Bank%20Use%20Only,for%20investments%20in%20equity%20securities.
    - detail on financial statement rubrics: https://www.federalreserve.gov/reportforms/forms/FR_Y-9LP20220705_i.pdf 

- FR Y-9SP (Parent Company Only Financial Statements for Small Holding Companies) - only reports data semi-annually 
    - financial statement: https://www.federalreserve.gov/apps/reportingforms/Download/DownloadAttachment?guid=9df5bb23-468c-4407-a874-7beaba03b930
    - detail on financial statement rubrics: https://www.federalreserve.gov/apps/reportingforms/Download/DownloadAttachment?guid=05505b4f-780d-49e0-a123-59e7cbf29afb


In [4]:
### Dataset has +2k cols
### Imported data dictionary with manually mapped cols identified as important, based on the following analyses
### These cols can have financial information (df_cols) or structural information (df_cols_meta)
path_data_dict_original = r"C:\Users\joaof\OneDrive\Education\University\PhD_UVM\Courses\202526\Fall\CS5870A_DataScienceI\Project\DSE-Project\Data_Processing\frozen\Financial_Download_Dictionary.xlsx"
# df_cols_data=pd.read_excel(convert_to_wsl_path(path_data_dict_original), sheet_name='Financial', engine='openpyxl')
df_cols_data = pd.read_excel(
    path_data_dict_original, sheet_name="Financial", engine="openpyxl"
)

In [5]:
df_cols_meta_original = pd.read_excel(
    path_data_dict_original, sheet_name="Structure", engine="openpyxl"
)
# df_cols_meta_original=pd.read_excel(convert_to_wsl_path(path_data_dict_original), sheet_name='Structure', engine='openpyxl')

In [6]:
df_cols_meta = df_cols_meta_original.rename(columns={"Item Name": "Short Description"})

### Filter database for selected columns

In [7]:
##Merge structural and financial columns into a unique df of cols

cond_cols_to_consider_finance = df_cols_data["To consider"] == True
cond_cols_to_consider_meta = df_cols_meta["to consider"] == True
df_cols_aux = pd.concat(
    [
        df_cols_data[cond_cols_to_consider_finance],
        df_cols_meta[cond_cols_to_consider_meta],
    ],
    axis=0,
).reset_index(drop=True)


df_cols_final = df_cols_aux[
    ["MDRM Item", "Start Date", "End Date", "Short Description"]
]
dict_cols_final = dict(
    zip(df_cols_final["MDRM Item"], df_cols_final["Short Description"])
)
df_cols_final

,MDRM Item,Start Date,End Date,Short Description
0,BHCA7204,2014-03-31,9999-12-31 00:00:00,TIER 1 LEVERAGE RATIO (ITEM 26 DIVIDED BY ITEM...
1,BHCA7205,2014-03-31,9999-12-31 00:00:00,TOTAL CAPITAL RATIO (ITEM 35.A DIVIDED BY ITEM...
2,BHCA7206,2014-03-31,9999-12-31 00:00:00,TIER 1 CAPITAL RATIO (ITEM 26 DIVIDED BY ITEM ...
3,BHCAH311,2016-03-31,9999-12-31 00:00:00,INSTITUTION-SPECIFIC CAPITAL BUFFER NECESSARY ...
4,BHCAP793,2014-03-31,9999-12-31 00:00:00,COMMON EQUITY TIER 1 CAPITAL RATIO (ITEM 19 DI...
...,...,...,...,...
175,RSSD9061,NaT,NaN,REASON FOR TERMINATION OF AN ENTITY
176,RSSD9132,NaT,NaN,PRIMARY ACTIVITY CODE
177,RSSD9200,NaT,NaN,ABBREVIATED STATE NAME
178,RSSD9425,NaT,NaN,BANK TYPE ANALYSIS CODE


In [8]:
## consolidated quarterly dfs into a single df with required cols (previously identified). Consolidate all dfs with +2000 would break the kernel.
df_all = pd.DataFrame()

for date, df in dfs.items():
    df_all = pd.concat(
        [
            df_all,
            df.loc[
                :,
                [col for col in dict_cols_final.keys() if col in df.columns]
                + ["quarter"],
            ],
        ],
        axis=0,
    )

df_all_final = df_all.reset_index(drop=True)

# Column/Feature engineering

### Total revenue

In [9]:
## add revenue col
### depending on the report revenue corresponded to one more multiple cols (represented by the codes, eg.: BHCK4107)
cols_revenue_FRY9C = ["BHCK4107", "BHCK4079", "BHCK3521", "BHCK3196"]
cols_revenue_FRY9SP = ["BHSP4000"]
cols_revenue_FRY9LP = ["BHCP4000"]

df_all_final[cols_revenue_FRY9C] = df_all_final[cols_revenue_FRY9C].astype(float)
df_all_final[cols_revenue_FRY9SP] = df_all_final[cols_revenue_FRY9SP].astype(float)
df_all_final[cols_revenue_FRY9LP] = df_all_final[cols_revenue_FRY9LP].astype(float)

df_all_final["REVENUE_FRY9C"] = df_all_final.loc[:, [*dict_cols_final, "quarter"]][
    cols_revenue_FRY9C
].sum(axis=1)
df_all_final["REVENUE_FRY9SP"] = df_all_final.loc[:, [*dict_cols_final, "quarter"]][
    cols_revenue_FRY9SP
].sum(axis=1)
df_all_final["REVENUE_FRY9LP"] = df_all_final.loc[:, [*dict_cols_final, "quarter"]][
    cols_revenue_FRY9LP
].sum(axis=1)


print(f"cols_revenue_FRY9C")
for key, value in {
    key: val for key, val in dict_cols_final.items() if key in cols_revenue_FRY9C
}.items():
    print(f"{key}: {value}")

cols_revenue_FRY9C
BHCK3196: REALIZED GAINS (LOSSES) ON AVAILABLE-FOR-SALE SECURITIES (BHC CONSOLIDATED)
BHCK3521: REALIZED GAINS (LOSSES) ON HELD-TO-MATURITY SECURITIES (BHC CONSOLIDATED)
BHCK4079: TOTAL NONINTEREST INCOME (BHC CONSOLIDATED)
BHCK4107: TOTAL INTEREST INCOME (BHC CONSOLIDATED)


Conclusion: Consider cols_revenue_FRY9LP, cols_revenue_FRY9SP and cols_revenue_FRY9C as values for col

#### Analysis of Revenue column to understand key features

In [10]:
## check revenue cols: at least total revenue from one report
### some cols do not have value, which was unexpected and requires further analysis

cond_REVENUE_FRY9C_null = df_all_final["REVENUE_FRY9C"] == 0
cond_REVENUE_FRY9SP_null = df_all_final["REVENUE_FRY9SP"] == 0
cond_REVENUE_FRY9LP_null = df_all_final["REVENUE_FRY9LP"] == 0

aux = df_all_final[
    cond_REVENUE_FRY9C_null & cond_REVENUE_FRY9SP_null & cond_REVENUE_FRY9LP_null
]
print(aux.shape[0], aux.shape[0] == 0)

37819 False


In [11]:
## check if one type of revenue exist the other does not exist (TRUE, TRUE corresponds when both do not exist)
(df_all_final[["REVENUE_FRY9SP", "REVENUE_FRY9LP"]] == 0).value_counts()

# Hypothesis moving foward: this can be just random individual quarters where banks did not report. it is important to check the ones that never reported information

REVENUE_FRY9SP  REVENUE_FRY9LP
False           True              159204
True            False             109874
                True               47812
Name: count, dtype: int64

In [12]:
# check if there are entities that never get SP or LP

never_sp_lp = df_all_final.groupby("RSSD9001").apply(
    lambda x: (x["REVENUE_FRY9SP"].eq(0) & x["REVENUE_FRY9LP"].eq(0)).all()
)

never_sp_lp_entities = never_sp_lp[never_sp_lp].index
never_sp_lp_entities

C:\Users\joaof\AppData\Local\Temp\ipykernel_37276\1502862191.py:3: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  never_sp_lp = df_all_final.groupby("RSSD9001").apply(


Index(['1055454', '1057421', '1063936', '1064072', '1064764', '1065257',
       '1066861', '1070934', '1081257', '1096747',
       ...
       '5924736', '5926918', '5930577', '5952551', '5975167', '5975952',
       '6005472', '6006899', '6010430', '6050100'],
      dtype='object', name='RSSD9001', length=480)

In [13]:
##print some of the values in never_sp_lp_entities
cond_ = df_all_final["RSSD9001"] == "2460569"
df_all_final[df_all_final["RSSD9001"].isin(never_sp_lp_entities) & cond_].loc[
    :,
    ["RSSD9001", "RSSD9017", "quarter"]
    + [col for col in df_all_final.columns if "REVENUE" in col],
].sort_values(by=["RSSD9001", "quarter"])

,RSSD9001,RSSD9017,quarter,REVENUE_FRY9C,REVENUE_FRY9SP,REVENUE_FRY9LP
316548,2460569,"NVE BANCORP, MHC",20000331,8035.0,0.0,0.0
313680,2460569,"NVE BANCORP, MHC",20000630,16412.0,0.0,0.0
308551,2460569,"NVE BANCORP, MHC",20000930,25167.0,0.0,0.0
305579,2460569,"NVE BANCORP, MHC",20001231,34338.0,0.0,0.0
300499,2460569,"NVE BANCORP, MHC",20010331,9038.0,0.0,0.0
...,...,...,...,...,...,...
23598,2460569,"NVE BANCORP, MHC",20221231,0.0,0.0,0.0
19244,2460569,"NVE BANCORP, MHC",20230630,0.0,0.0,0.0
14913,2460569,"NVE BANCORP, MHC",20231231,0.0,0.0,0.0
10609,2460569,"NVE BANCORP, MHC",20240630,0.0,0.0,0.0


In [14]:
# check if the banks in never_sp_lp_entities appear in lot of quarters
df_all_final[df_all_final["RSSD9001"].isin(never_sp_lp_entities)].groupby("RSSD9001")[
    "quarter"
].nunique().sort_values()

RSSD9001
1126961      1
1128808      1
2128485      1
1140453      1
1143258      1
          ... 
1142439     72
2460550     78
2460569     78
2589723    101
2589714    101
Name: quarter, Length: 480, dtype: int64

In [15]:
# check if the banks in never_sp_lp_entities report instead revenue through Y9C
df_all_final[df_all_final["RSSD9001"].isin(never_sp_lp_entities)].groupby("RSSD9001")[
    "REVENUE_FRY9C"
].apply(lambda s: (s != 0).sum()).sort_values().describe()

count    480.000000
mean       2.318750
std        9.053567
min        0.000000
25%        0.000000
50%        0.000000
75%        0.000000
max      101.000000
Name: REVENUE_FRY9C, dtype: float64

In [16]:
(
    df_all_final[df_all_final["RSSD9001"].isin(never_sp_lp_entities)]
    .groupby("RSSD9001")["REVENUE_FRY9C"]
    .apply(lambda s: (s != 0).sum())
    .sort_values()
    > 0
).value_counts()

REVENUE_FRY9C
False    418
True      62
Name: count, dtype: int64

Conclusion: Remove cols Y9C cols and rows RSSD9001 in never_sp_lp_entities (entities that never had sp or lp values for revenue)


In [17]:
df_all_final[
    (df_all_final["REVENUE_FRY9C"] != 0) & (df_all_final["REVENUE_FRY9LP"] == 0)
].loc[
    :,
    ["RSSD9001", "quarter"] + [col for col in df_all_final.columns if "REVENUE" in col],
].sort_values(
    by=["RSSD9001", "quarter"]
)

,RSSD9001,quarter,REVENUE_FRY9C,REVENUE_FRY9SP,REVENUE_FRY9LP
233938,1021570,20050331,2400.0,0.0,0.0
298731,1021879,20010331,4766.0,0.0,0.0
282491,1022166,20020331,3578.0,0.0,0.0
276642,1022166,20020630,7107.0,0.0,0.0
274271,1022166,20020930,10423.0,0.0,0.0
...,...,...,...,...,...
8452,5806739,20240930,148447.0,0.0,0.0
7955,5806739,20241231,200508.0,0.0,0.0
4198,5806739,20250331,52151.0,0.0,0.0
3672,5806739,20250630,105306.0,0.0,0.0


In [18]:
## Parent companies that are also holding companies have to both report FRY9C and FRY9LP
## (if other subsidiaries have negative revenue the holding can have lower revenue than the parent)
## In FRY9C one dimension of revenue corresponds to "gains or losses of sale of specific assets". One reason to have unexpected negative values of revenue
cond_REVENUE_FRY9C_nonnull = df_all_final["REVENUE_FRY9C"] != 0
cond_REVENUE_FRY9LP_max = (
    df_all_final[["REVENUE_FRY9C", "REVENUE_FRY9SP", "REVENUE_FRY9LP"]]
    .abs()
    .idxmax(axis=1)
) == "REVENUE_FRY9LP"
cond_REVENUE_FRY9C_max = (
    df_all_final[["REVENUE_FRY9C", "REVENUE_FRY9SP", "REVENUE_FRY9LP"]]
    .abs()
    .idxmax(axis=1)
) == "REVENUE_FRY9C"
cond_REVENUE_FRY9SP_max = (
    df_all_final[["REVENUE_FRY9C", "REVENUE_FRY9SP", "REVENUE_FRY9LP"]]
    .abs()
    .idxmax(axis=1)
) == "REVENUE_FRY9SP"

df_all_final[(cond_REVENUE_FRY9C_nonnull) & cond_REVENUE_FRY9LP_max][
    [
        "RSSD9001",
        "RSSD9017",
        "quarter",
        "REVENUE_FRY9C",
        "REVENUE_FRY9SP",
        "REVENUE_FRY9LP",
    ]
    + ["BHCK4107", "BHCK4079", "BHCK3521", "BHCK3196"]
    + ["BHCK8560", "BHCK8561", "BHCKB496"]
].sort_values(by=["RSSD9017", "quarter"])

,RSSD9001,RSSD9017,quarter,REVENUE_FRY9C,REVENUE_FRY9SP,REVENUE_FRY9LP,BHCK4107,BHCK4079,BHCK3521,BHCK3196,BHCK8560,BHCK8561,BHCKB496
4204,5902912,"ADAM CORPORATION/GROUP, THE",20250331,15589.0,0.0,31440.0,39926.0,-24337.0,0.0,0.0,433,0,-4
251938,1246926,"ALBANY BANCSHARES, INC.",20040331,2915.0,0.0,3160.0,2694.0,221.0,0.0,0.0,0,0,0
235118,1246926,"ALBANY BANCSHARES, INC.",20050331,2944.0,0.0,3020.0,2734.0,210.0,0.0,0.0,0,0,0
108040,2533100,"ALIKAT INVESTMENTS, INC.",20140630,4831.0,0.0,5301.0,4753.0,78.0,0.0,0.0,0,-460,0
104345,2533100,"ALIKAT INVESTMENTS, INC.",20140930,8937.0,0.0,11819.0,7064.0,1873.0,0.0,0.0,0,-486,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
147505,1210066,"WEST BANCORPORATION, INC.",20110630,31660.0,0.0,38670.0,26968.0,4692.0,0.0,0.0,456,-203,13
234485,1109227,"WESTERN COMMERCE BANCSHARES OF CARLSBAD, INC.",20050331,4252.0,0.0,4380.0,3722.0,530.0,0.0,0.0,0,0,-1
138632,3248513,"WILSHIRE BANCORP, INC.",20120331,36073.0,0.0,62171.0,29266.0,6804.0,0.0,3.0,758,441,-1
110685,1399671,"WNB BANCSHARES, INC.",20140331,9570.0,0.0,20725.0,8672.0,898.0,0.0,0.0,0,0,-16


In [19]:
# understand if there are cases where 9LP and 9SP is filled by the same company in the same quarter
col1 = "REVENUE_FRY9SP"
col2 = "REVENUE_FRY9LP"

both_non_zero = (df_all_final[col1] != 0) & (df_all_final[col2] != 0)

# Check if it ever happens
both_non_zero.any()

np.False_

In [20]:
# Identify if when focusing on LP and SP values, net income and revenue can have weird values (negative revenues)
print((df_all_final.loc[:, "REVENUE_FRY9LP"] < 0).value_counts())
cond_neg_rev = df_all_final.loc[:, "REVENUE_FRY9LP"] < 0
idx_neg_rev = df_all_final[cond_neg_rev]["RSSD9001"].unique()
df_all_final[df_all_final["RSSD9001"].isin(idx_neg_rev[2:3])].loc[
    :,
    [
        "RSSD9017",
        "REVENUE_FRY9C",
        "REVENUE_FRY9LP",
        "REVENUE_FRY9SP",
        "quarter",
        "RSSD9008",
    ],
]
# answer: it is normal, there are gains(losses) rubrics in LP, that can be negative

REVENUE_FRY9LP
False    316152
True        738
Name: count, dtype: int64


,RSSD9017,REVENUE_FRY9C,REVENUE_FRY9LP,REVENUE_FRY9SP,quarter,RSSD9008
3396,CIBC BANCORP USA INC.,2852855.0,-115689.0,0.0,20250630,99991231
4173,CIBC BANCORP USA INC.,1376424.0,16573.0,0.0,20250331,99991231
7673,CIBC BANCORP USA INC.,5909092.0,320355.0,0.0,20241231,99991231
8427,CIBC BANCORP USA INC.,4404818.0,137375.0,0.0,20240930,99991231
11959,CIBC BANCORP USA INC.,2851779.0,149276.0,0.0,20240630,99991231
12711,CIBC BANCORP USA INC.,1510471.0,102411.0,0.0,20240331,99991231
16285,CIBC BANCORP USA INC.,5261388.0,247412.0,0.0,20231231,99991231
17020,CIBC BANCORP USA INC.,3791601.0,236597.0,0.0,20230930,99991231
20638,CIBC BANCORP USA INC.,2478989.0,88229.0,0.0,20230630,99991231
21354,CIBC BANCORP USA INC.,1200794.0,72681.0,0.0,20230331,99991231


In [21]:
# Identify if when focusing on LP and SP values, net income and revenue can have weird values (negative revenues)
print((df_all_final.loc[:, "REVENUE_FRY9SP"] < 0).value_counts())
cond_neg_rev = df_all_final.loc[:, "REVENUE_FRY9SP"] < 0
idx_neg_rev = (
    df_all_final[cond_neg_rev].sort_values("REVENUE_FRY9SP")["RSSD9001"].unique()
)
df_all_final[df_all_final["RSSD9001"].isin(idx_neg_rev[:1])].loc[
    :,
    [
        "RSSD9017",
        "REVENUE_FRY9C",
        "REVENUE_FRY9LP",
        "REVENUE_FRY9SP",
        "quarter",
        "RSSD9008",
        "BHSP0447",
    ],
]
# answer: it is normal, there are gains(losses) and other income rubrics in SP, that can be negative.
# Important to notice that subsidiary related income is sales between major parent and subsidiary, not consolidated revenues from subsidiary

REVENUE_FRY9SP
False    316743
True        147
Name: count, dtype: int64


,RSSD9017,REVENUE_FRY9C,REVENUE_FRY9LP,REVENUE_FRY9SP,quarter,RSSD9008,BHSP0447
77450,"SEAWAY BANCSHARES, INC.",0.0,0.0,-38121.0,20161231,20170127,0
83013,"SEAWAY BANCSHARES, INC.",0.0,0.0,0.0,20160630,20170127,0
88653,"SEAWAY BANCSHARES, INC.",0.0,0.0,0.0,20151231,20170127,0
94357,"SEAWAY BANCSHARES, INC.",0.0,0.0,-1176.0,20150630,20170127,0
100212,"SEAWAY BANCSHARES, INC.",31485.0,1.0,0.0,20141231,20170127,NaN
104011,"SEAWAY BANCSHARES, INC.",25169.0,1.0,0.0,20140930,20170127,NaN
106702,"SEAWAY BANCSHARES, INC.",16608.0,1.0,0.0,20140630,20170127,NaN
110582,"SEAWAY BANCSHARES, INC.",8768.0,1.0,0.0,20140331,20170127,NaN
113332,"SEAWAY BANCSHARES, INC.",35837.0,2016.0,0.0,20131231,20170127,NaN
117284,"SEAWAY BANCSHARES, INC.",27749.0,2015.0,0.0,20130930,20170127,NaN


### Net Income

In [22]:
## add netincome col. followed same rational as with revenue
cols_netincome_FRY9C = ["BHCK4340"]
cols_netincome_FRY9SP = ["BHSP4340"]
cols_netincome_FRY9LP = ["BHCP4340"]

df_all_final[cols_netincome_FRY9C] = df_all_final[cols_netincome_FRY9C].astype(float)
df_all_final[cols_netincome_FRY9SP] = df_all_final[cols_netincome_FRY9SP].astype(float)
df_all_final[cols_netincome_FRY9LP] = df_all_final[cols_netincome_FRY9LP].astype(float)

df_all_final["NET_INCOME_FRY9C"] = df_all_final.loc[:, [*dict_cols_final, "quarter"]][
    cols_netincome_FRY9C
].sum(axis=1)
df_all_final["NET_INCOME_FRY9SP"] = df_all_final.loc[:, [*dict_cols_final, "quarter"]][
    cols_netincome_FRY9SP
].sum(axis=1)
df_all_final["NET_INCOME_FRY9LP"] = df_all_final.loc[:, [*dict_cols_final, "quarter"]][
    cols_netincome_FRY9LP
].sum(axis=1)


print(f"cols_netincome_FRY9C")
for key, value in {
    key: val for key, val in dict_cols_final.items() if key in cols_netincome_FRY9C
}.items():
    print(f"{key}: {value}")

cols_netincome_FRY9C
BHCK4340: NET INCOME (LOSS) ATTRIBUTABLE TO BANK HOLDING COMPANY (BHC CONSOLIDATED)


#### Analysis of Net Income column(s) to understand key features

In [23]:
## check revenue cols: at least total revenue from one report

cond_NET_INCOME_FRY9C_null = df_all_final["NET_INCOME_FRY9C"] == 0
cond_NET_INCOME_FRY9SP_null = df_all_final["NET_INCOME_FRY9SP"] == 0
cond_NET_INCOME_FRY9LP_null = df_all_final["NET_INCOME_FRY9LP"] == 0

aux = df_all_final[
    cond_NET_INCOME_FRY9C_null
    & cond_NET_INCOME_FRY9SP_null
    & cond_NET_INCOME_FRY9LP_null
]
print(aux.shape[0], aux.shape[0] == 0)

791 False


In [24]:
## check if one type of revenue exist the other does not exist (TRUE, TRUE corresponds when both do not exist)
(df_all_final[["NET_INCOME_FRY9SP", "NET_INCOME_FRY9LP"]] == 0).value_counts()

NET_INCOME_FRY9SP  NET_INCOME_FRY9LP
False              True                 190774
True               False                125249
                   True                    867
Name: count, dtype: int64

In [25]:
# Identify if when focusing on LP and SP values, net income and revenue can have weird values (net income larger than revenue)
print(
    (
        df_all_final.loc[:, "NET_INCOME_FRY9SP"] > df_all_final.loc[:, "REVENUE_FRY9SP"]
    ).value_counts()
)
cond_neg_rev = (
    df_all_final.loc[:, "NET_INCOME_FRY9SP"] > df_all_final.loc[:, "REVENUE_FRY9SP"]
)
idx_neg_rev = (
    df_all_final[cond_neg_rev].sort_values("REVENUE_FRY9SP")["RSSD9001"].unique()
)
df_all_final[df_all_final["RSSD9001"].isin(idx_neg_rev[:1])].loc[
    :,
    [
        "RSSD9017",
        "NET_INCOME_FRY9SP",
        "REVENUE_FRY9SP",
        "quarter",
        "RSSD9008",
        "BHSP0447",
    ],
]
# answer: one potential driver off this is the equity rubric, where right before the net income rubric not distributed profits from subsdiaries can enter as new
# source of profit

False    167396
True     149494
Name: count, dtype: int64


,RSSD9017,NET_INCOME_FRY9SP,REVENUE_FRY9SP,quarter,RSSD9008,BHSP0447
83367,"DELAWARE BANCSHARES, INC.",162.0,410.0,20160630,20160731,4
89012,"DELAWARE BANCSHARES, INC.",591.0,489.0,20151231,20160731,7
94720,"DELAWARE BANCSHARES, INC.",240.0,244.0,20150630,20160731,3
100580,"DELAWARE BANCSHARES, INC.",967.0,-8156.0,20141231,20160731,6
107076,"DELAWARE BANCSHARES, INC.",596.0,34.0,20140630,20160731,3
113707,"DELAWARE BANCSHARES, INC.",1309.0,1050.0,20131231,20160731,7
120463,"DELAWARE BANCSHARES, INC.",633.0,520.0,20130630,20160731,3
127335,"DELAWARE BANCSHARES, INC.",1343.0,940.0,20121231,20160731,7
134272,"DELAWARE BANCSHARES, INC.",631.0,470.0,20120630,20160731,4
141286,"DELAWARE BANCSHARES, INC.",1504.0,1106.0,20111231,20111231,6


### Net income, before undistributed income from subsidiaries

In [26]:
## add netincome col. followed same rational as with revenue
cols_netincome_before_undis_rub_FRY9SP = ["BHSP0496"]
cols_netincome_before_undis_rub_FRY9LP = ["BHCP0496"]

df_all_final[cols_netincome_before_undis_rub_FRY9SP] = df_all_final[
    cols_netincome_before_undis_rub_FRY9SP
].astype(float)
df_all_final[cols_netincome_before_undis_rub_FRY9LP] = df_all_final[
    cols_netincome_before_undis_rub_FRY9LP
].astype(float)

df_all_final["NET_INCOME_BEF_UND_RUB_FRY9SP"] = df_all_final.loc[
    :, [*dict_cols_final, "quarter"]
][cols_netincome_before_undis_rub_FRY9SP].sum(axis=1)
df_all_final["NET_INCOME_BEF_UND_RUB_FRY9LP"] = df_all_final.loc[
    :, [*dict_cols_final, "quarter"]
][cols_netincome_before_undis_rub_FRY9LP].sum(axis=1)

#### Analysis of Net income, before undistributed income from subsidiaries column(s) to understand key features

In [27]:
# Identify if when focusing on LP and SP values, net income BEFORE UNDIST RUBRIBS and revenue can have weird values (net income larger than revenue)
print(
    (
        df_all_final.loc[:, "NET_INCOME_BEF_UND_RUB_FRY9SP"]
        > df_all_final.loc[:, "REVENUE_FRY9SP"]
    ).value_counts()
)
cond_neg_rev = (
    df_all_final.loc[:, "NET_INCOME_BEF_UND_RUB_FRY9SP"]
    > df_all_final.loc[:, "REVENUE_FRY9SP"]
)
idx_neg_rev = (
    df_all_final[cond_neg_rev].sort_values("REVENUE_FRY9SP")["RSSD9001"].unique()
)
df_all_final[df_all_final["RSSD9001"].isin(idx_neg_rev[:1])].loc[
    :,
    [
        "RSSD9017",
        "NET_INCOME_BEF_UND_RUB_FRY9SP",
        "REVENUE_FRY9SP",
        "quarter",
        "RSSD9008",
        "BHSP0447",
    ],
]
# answer: other non operating revenues also have a big impact

False    314674
True       2216
Name: count, dtype: int64


,RSSD9017,NET_INCOME_BEF_UND_RUB_FRY9SP,REVENUE_FRY9SP,quarter,RSSD9008,BHSP0447
164968,"CRESCENT CAPITAL VI, L.L.C.",0.0,0.0,20100331,20100625,NaN
170328,"CRESCENT CAPITAL VI, L.L.C.",0.0,0.0,20091231,20100625,NaN
171680,"CRESCENT CAPITAL VI, L.L.C.",0.0,0.0,20090930,20100625,NaN
177142,"CRESCENT CAPITAL VI, L.L.C.",0.0,0.0,20090630,20100625,NaN
178478,"CRESCENT CAPITAL VI, L.L.C.",0.0,0.0,20090331,20100625,NaN
184012,"CRESCENT CAPITAL VI, L.L.C.",-2962.0,-3399.0,20081231,20100625,-3399


### Total Assets

In [28]:
# LP:BHCP2170
# SP:BHSP2170
# 9C:BHCK2170

In [29]:
## add total_assets col
### depending on the report total_assets corresponded to one more multiple cols (represented by the codes, eg.: BHCK4107)
cols_total_assets_FRY9C = ["BHCK2170"]
cols_total_assets_FRY9SP = ["BHSP2170"]
cols_total_assets_FRY9LP = ["BHCP2170"]

df_all_final[cols_total_assets_FRY9C] = df_all_final[cols_total_assets_FRY9C].astype(
    float
)
df_all_final[cols_total_assets_FRY9SP] = df_all_final[cols_total_assets_FRY9SP].astype(
    float
)
df_all_final[cols_total_assets_FRY9LP] = df_all_final[cols_total_assets_FRY9LP].astype(
    float
)

df_all_final["TOTAL_ASSETS_FRY9C"] = df_all_final.loc[:, [*dict_cols_final, "quarter"]][
    cols_total_assets_FRY9C
].sum(axis=1)
df_all_final["TOTAL_ASSETS_FRY9SP"] = df_all_final.loc[
    :, [*dict_cols_final, "quarter"]
][cols_total_assets_FRY9SP].sum(axis=1)
df_all_final["TOTAL_ASSETS_FRY9LP"] = df_all_final.loc[
    :, [*dict_cols_final, "quarter"]
][cols_total_assets_FRY9LP].sum(axis=1)


print(f"cols_total_assets_FRY9C")
for key, value in {
    key: val for key, val in dict_cols_final.items() if key in cols_total_assets_FRY9C
}.items():
    print(f"{key}: {value}")

cols_total_assets_FRY9C
BHCK2170: TOTAL ASSETS (BHC CONSOLIDATED)


### Total Equity

In [30]:
# LP:BHCP3210
# SP:BHSP3210
# 9C:BHCKG105

In [31]:
## add total_equity col
### depending on the report total_equity corresponded to one more multiple cols (represented by the codes, eg.: BHCK4107)
cols_total_equity_FRY9C = ["BHCKG105"]
cols_total_equity_FRY9SP = ["BHSP3210"]
cols_total_equity_FRY9LP = ["BHCP3210"]

df_all_final[cols_total_equity_FRY9C] = df_all_final[cols_total_equity_FRY9C].astype(
    float
)
df_all_final[cols_total_equity_FRY9SP] = df_all_final[cols_total_equity_FRY9SP].astype(
    float
)
df_all_final[cols_total_equity_FRY9LP] = df_all_final[cols_total_equity_FRY9LP].astype(
    float
)

df_all_final["TOTAL_EQUITY_FRY9C"] = df_all_final.loc[:, [*dict_cols_final, "quarter"]][
    cols_total_equity_FRY9C
].sum(axis=1)
df_all_final["TOTAL_EQUITY_FRY9SP"] = df_all_final.loc[
    :, [*dict_cols_final, "quarter"]
][cols_total_equity_FRY9SP].sum(axis=1)
df_all_final["TOTAL_EQUITY_FRY9LP"] = df_all_final.loc[
    :, [*dict_cols_final, "quarter"]
][cols_total_equity_FRY9LP].sum(axis=1)


print(f"cols_total_equity_FRY9C")
for key, value in {
    key: val for key, val in dict_cols_final.items() if key in cols_total_equity_FRY9C
}.items():
    print(f"{key}: {value}")

cols_total_equity_FRY9C
BHCKG105: TOTAL EQUITY CAPITAL (BHC CONSOLIDATED)


### Earnings before taxes columns

In [32]:
# LP:BHCP4250
# SP:BHSP4250
# 9C:BHCK4301

In [33]:
## add ebt col
### depending on the report ebt corresponded to one more multiple cols (represented by the codes, eg.: BHCK4107)
cols_ebt_FRY9C = ["BHCK4301"]
cols_ebt_FRY9SP = ["BHSP4250"]
cols_ebt_FRY9LP = ["BHCP4250"]

df_all_final[cols_ebt_FRY9C] = df_all_final[cols_ebt_FRY9C].astype(float)
df_all_final[cols_ebt_FRY9SP] = df_all_final[cols_ebt_FRY9SP].astype(float)
df_all_final[cols_ebt_FRY9LP] = df_all_final[cols_ebt_FRY9LP].astype(float)

df_all_final["EBT_FRY9C"] = df_all_final.loc[:, [*dict_cols_final, "quarter"]][
    cols_ebt_FRY9C
].sum(axis=1)
df_all_final["EBT_FRY9SP"] = df_all_final.loc[:, [*dict_cols_final, "quarter"]][
    cols_ebt_FRY9SP
].sum(axis=1)
df_all_final["EBT_FRY9LP"] = df_all_final.loc[:, [*dict_cols_final, "quarter"]][
    cols_ebt_FRY9LP
].sum(axis=1)


print(f"cols_ebt_FRY9C")
for key, value in {
    key: val for key, val in dict_cols_final.items() if key in cols_ebt_FRY9C
}.items():
    print(f"{key}: {value}")

cols_ebt_FRY9C
BHCK4301: Income (loss) before applicable income taxes and discontinued operations (sum of items 8.a and 8.b)


### Capital (Tier 1 + Tier 2) - proxy

In [34]:
### SP: include all equity rubrics. try to deduct goodwill, intangile assets (validate sources) (ONLY HAVE GOODWILL) BHSP3238, 0087, 0202
### LP: include all equity rubrics. try to deduct goodwill, intangile assets (validate sources) BHCP3238, 4485, 0087,0536, 0202, 0203, 3163, 3164, 3165
### 9C: include all equity rubrics. try to deduct goodwill, intangile assets (validate sources)  BHCK3164

## To check:
### 9C: Noncontrolling (minority) interests in consolidated subsidiaries, subordinated debt (potentially to be excluded)

In [35]:
## add capital_tier12 col
### depending on the report capital_tier12 corresponded to one more multiple cols (represented by the codes, eg.: BHCK4107)
cols_capital_tier12_FRY9C = ["BHCK3164"]
cols_capital_tier12_FRY9SP = ["BHSP3238", "BHSP0087", "BHSP0202"]
cols_capital_tier12_FRY9LP = [
    "BHCP3238",
    "BHCP4485",
    "BHCP0087",
    "BHCP0536",
    "BHCP0202",
    "BHCP0203",
    "BHCP3163",
    "BHCP3164",
    "BHCP3165",
]

df_all_final[cols_capital_tier12_FRY9C] = df_all_final[
    cols_capital_tier12_FRY9C
].astype(float)
df_all_final[cols_capital_tier12_FRY9SP] = df_all_final[
    cols_capital_tier12_FRY9SP
].astype(float)
df_all_final[cols_capital_tier12_FRY9LP] = df_all_final[
    cols_capital_tier12_FRY9LP
].astype(float)

df_all_final["CAPITAL_TIER12_FRY9C"] = df_all_final.loc[
    :, [*dict_cols_final, "quarter"]
][cols_capital_tier12_FRY9C].sum(axis=1)
df_all_final["CAPITAL_TIER12_FRY9SP"] = df_all_final.loc[
    :, [*dict_cols_final, "quarter"]
][cols_capital_tier12_FRY9SP].sum(axis=1)
df_all_final["CAPITAL_TIER12_FRY9LP"] = df_all_final.loc[
    :, [*dict_cols_final, "quarter"]
][cols_capital_tier12_FRY9LP].sum(axis=1)


print(f"cols_capital_tier12_FRY9C")
for key, value in {
    key: val for key, val in dict_cols_final.items() if key in cols_capital_tier12_FRY9C
}.items():
    print(f"{key}: {value}")

cols_capital_tier12_FRY9C
BHCK3164: MORTGAGE SERVICING ASSETS (BHC CONSOLIDATED)


### Risk-adjust assets: inputs

In [36]:
###SP - BHSP:
## Cash and due from depository institutions (5993)
## Securities (0390)
## Loans and leases, held for investment and held for sale, net of the allowance (item 3.a minus 3.b). (2723)
## Equity investment (3239 + 0088 + 0201)
## Goodwill (3238 + 0087 + 0202)
## Loans and advances to and receivables due from bank subsidiary (3128)
## Loans and advances to and receivables due from non bank subsidiary (0089)
## Loans and advances to and receivables due from subsidiary holding company (3523)
## Other assets (0027)
## Balances due from related nonbank companies (other than investments)5 (3620)


###LP - BHCP:
## Cash and balances due from depository institutions: a. Balances with subsidiary or affiliated depository institutions (5993)
## Cash and balances due from depository institutions: b. Balances with unrelated depository institutions (0010)
## Securities: U.S. Treasury securities (0400)
## Securities: b. Securities of U.S. Government agencies and corporations and securities issued by states and political subdivisions (6791)
## Securities: c. Other debt and equity securities (1299)
## Securities purchased under agreements to resell (0277)
### f. Loans and leases, held for investment and held for sale, net of allowance (sum of items 4.c and 4.d minus item 4.e) (2125)

## Investments in and receivables due from subsidiaries and associated companies (0365)

## Premises and fixed assets (including right-of-use assets) (2145)

## Intangible assets (other than reported in item 5 above)
### a. Goodwill (3163)
### b. Mortgage servicing assets (3164)
### c. Other identifiable intangibles (3165)

## Other assets (2160)

## Balances due from related institutions, other than investments
### a. Related banks (3602)
### b. Related nonbank companies (3603)
### c. Related holding companies (3604)

## TOTAL ASSETS (sum of items 1.a through 3, and 4.f minus 9.c above) (2170)


###9C - BHCK:
## Noninterest-bearing balances and currency and coin1 (0081)
## Interest-bearing balances: In U.S. offices (0395)
## Interest-bearing balances: In foreign offices, Edge and Agreement subsidiaries, and IBFs (0397)
## Securities: Held-to-maturity securities (from Schedule HC-B, column A) (1754)
## Securities: Available-for-sale securities (from Schedule HC-B, column D) (1773)
## Securities: Equity securities with readily determinable fair values not held for trading (JA22)
## Federal funds sold and securities purchased under agreements to resell:
## Federal funds sold and securities purchased under agreements to resell: Federal funds sold in domestic offices (B987)
## Federal funds sold and securities purchased under agreements to resell: Securities purchased under agreements to resell (B989)
## Loans and lease financing receivables: Loans and leases, held for investment, net of allowance for loan and lease losses (B529)
## Trading assets (from Schedule HC-D) (3545)
## Premises and fixed assets (including capitalized leases) (2145)
## Other real estate owned (from Schedule HC-M) (2150)
## Investments in unconsolidated subsidiaries and associated companies (2130)
## Direct and indirect investments in real estate ventures (3656)
## Intangible assets (from Schedule HC-M) (2143)
## Other assets (from Schedule HC-F) (2160)

In [37]:
# Each financial-statement asset line is mapped to a Basel standardized credit-risk weight.
# Sources: https://en.wikipedia.org/wiki/Standardized_approach_(credit_risk),
#          https://www.bis.org/publ/bcbs128.pdf
# Exact RWA requires non-aggregated exposure data, which FS do not provide.
# Therefore each asset category is assigned a min–max weight range reflecting possible outcomes.
path_risk_weights = r"C:\Users\joaof\OneDrive\Education\University\PhD_UVM\Courses\202526\Fall\CS5870A_DataScienceI\Project\DSE-Project\Data_Processing\frozen\DSI_banking_weights.xlsx"
df_report_assets = pd.read_excel(path_risk_weights, sheet_name="report_assets")
df_bis_info = pd.read_excel(path_risk_weights, sheet_name="bis_info")

In [38]:
df_report_assets["RSSD9001"] = (
    df_report_assets["report_code_partI"] + df_report_assets["report_code_partII"]
)

In [39]:
df_report_assets["id_bis_min_adjusted"] = (
    df_report_assets["id_bis_min"].fillna(0).astype(int)
)
df_report_assets["id_bis_max_adjusted"] = (
    df_report_assets["id_bis_max"].fillna(0).astype(int)
)

In [40]:
df_report_assets["bis_weight_min"] = df_report_assets.merge(
    df_bis_info, left_on="id_bis_min_adjusted", right_on="id_bis", how="left"
)["Weight"]
df_report_assets["bis_weight_max"] = df_report_assets.merge(
    df_bis_info, left_on="id_bis_max_adjusted", right_on="id_bis", how="left"
)["Weight"]

In [41]:
df_report_assets["col_name_final"] = (
    df_report_assets["col_name"] + "_FRY9" + df_report_assets["report_type"]
)

In [42]:
for row in df_report_assets.iterrows():
    df_all_final[row[1]["col_name_final"]] = df_all_final[row[1]["RSSD9001"]].astype(
        float
    )

C:\Users\joaof\AppData\Local\Temp\ipykernel_37276\3751065440.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_all_final[row[1]["col_name_final"]] = df_all_final[row[1]["RSSD9001"]].astype(
C:\Users\joaof\AppData\Local\Temp\ipykernel_37276\3751065440.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_all_final[row[1]["col_name_final"]] = df_all_final[row[1]["RSSD9001"]].astype(
C:\Users\joaof\AppData\Local\Temp\ipykernel_37276\3751065440.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the resu

In [43]:
##CROSS CHECK: SUM OF ASSETS PER REPORT_TYPE SHOULD BE EQUAL TO TOTAL ASSETS
total_assets_aux_1_ = df_all_final["TOTAL_ASSETS_FRY9C"]
total_assets_aux_2_ = df_all_final[
    [
        col
        for col in df_all_final
        if "ASSETS_" in col and "FRY9C" in col and "TOTAL" not in col
    ]
]
print("FRY9C")
print(total_assets_aux_1_.sum() / 10**9, total_assets_aux_2_.sum().sum() / 10**9)
print("\n")
total_assets_aux_1_ = df_all_final["TOTAL_ASSETS_FRY9SP"]
total_assets_aux_2_ = df_all_final[
    [
        col
        for col in df_all_final
        if "ASSETS_" in col and "FRY9SP" in col and "TOTAL" not in col
    ]
]
print("FRY9SP")
print(total_assets_aux_1_.sum() / 10**9, total_assets_aux_2_.sum().sum() / 10**9)
print("\n")
total_assets_aux_1_ = df_all_final["TOTAL_ASSETS_FRY9LP"]
total_assets_aux_2_ = df_all_final[
    [
        col
        for col in df_all_final
        if "ASSETS_" in col and "FRY9LP" in col and "TOTAL" not in col
    ]
]
print("FRY9LP")
print(total_assets_aux_1_.sum() / 10**9, total_assets_aux_2_.sum().sum() / 10**9)
print("\n")
# Conclusions: The columns names were double checked, and the differences might arrive due to roundings

FRY9C
1807.764289861 1751.374817959


FRY9SP
5.082916564 5.070019843


FRY9LP
445.691795339 445.676456108




### Loan Loss Provisions 


In [44]:
# LP: BHCPJJ33
# SP: BHSP4093 (these include impairments in goodwills and other intangible assets)
# 9C: BHCT4230

In [45]:
## add ebt col
### depending on the report ebt corresponded to one more multiple cols (represented by the codes, eg.: BHCK4107)
cols_ebt_FRY9C = ["BHCT4230"]
cols_ebt_FRY9SP = ["BHSP4093"]
cols_ebt_FRY9SP_subtract = 0  # TBD INCLUDE PER QUARTER CHANGES IN [col for col in df_all_final.columns if ('GOODWILL' in col or 'INTANGIBLE'  in col) and 'FRY9SP' in col]
cols_ebt_FRY9LP = ["BHCPJJ33"]

df_all_final[cols_ebt_FRY9C] = df_all_final[cols_ebt_FRY9C].astype(float)
df_all_final[cols_ebt_FRY9SP] = df_all_final[cols_ebt_FRY9SP].astype(float)
df_all_final[cols_ebt_FRY9LP] = df_all_final[cols_ebt_FRY9LP].astype(float)

df_all_final["LOANS_LOSS_PROVISIONS_FRY9C"] = df_all_final.loc[
    :, [*dict_cols_final, "quarter"]
][cols_ebt_FRY9C].sum(axis=1)
df_all_final["LOANS_LOSS_PROVISIONS_FRY9SP"] = df_all_final.loc[
    :, [*dict_cols_final, "quarter"]
][cols_ebt_FRY9SP].sum(axis=1)
df_all_final["LOANS_LOSS_PROVISIONS_FRY9LP"] = df_all_final.loc[
    :, [*dict_cols_final, "quarter"]
][cols_ebt_FRY9LP].sum(axis=1)


print(f"cols_ebt_FRY9C")
for key, value in {
    key: val for key, val in dict_cols_final.items() if key in cols_ebt_FRY9C
}.items():
    print(f"{key}: {value}")

C:\Users\joaof\AppData\Local\Temp\ipykernel_37276\2558204679.py:12: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_all_final["LOANS_LOSS_PROVISIONS_FRY9C"] = df_all_final.loc[
C:\Users\joaof\AppData\Local\Temp\ipykernel_37276\2558204679.py:15: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_all_final["LOANS_LOSS_PROVISIONS_FRY9SP"] = df_all_final.loc[


cols_ebt_FRY9C
BHCT4230: PROVISION FOR LOAN AND LEASE LOSSES (MUST EQUAL SCHEDULE HI, ITEM 4)


C:\Users\joaof\AppData\Local\Temp\ipykernel_37276\2558204679.py:18: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_all_final["LOANS_LOSS_PROVISIONS_FRY9LP"] = df_all_final.loc[


### Charge-offs and Recoveries on Loans and Leases

In [46]:
# 9C - charged offs:
# loans secured by real estate: BHCT4230, BHCKC891, BHCKC893, BHCK3584, BHCK5411, BHCKC234 , BHCKC235, BHCK3588 , BHCKC895 , BHCKC897 , BHCKB512
# loans to farmers: BHCK4655
# loans to commerce and industry: BHCK4645, BHCK4646
# loans to households: BHCKB514, BHCKK129, BHCKK205
# 6 Loans to foreign governments and official institutions:	BHCK4643
# 7 Lease financing receivables:	BHCK4644
# lease to financial receivables: BHCKF185, BHCKC880

In [47]:
## add charged_off_columns
# Define charged-off columns for each category
cols_loans_realestate = [
    "BHCT4230",
    "BHCKC891",
    "BHCKC893",
    "BHCK3584",
    "BHCK5411",
    "BHCKC234",
    "BHCKC235",
    "BHCK3588",
    "BHCKC895",
    "BHCKC897",
    "BHCKB512",
]
cols_loans_farmers = ["BHCK4655"]
cols_loans_commerce_industry = ["BHCK4645", "BHCK4646"]
cols_loans_households = ["BHCKB514", "BHCKK129", "BHCKK205"]
cols_loans_foreign_gov = ["BHCK4643"]
cols_loans_lease_financing = ["BHCK4644"]
cols_loans_financial_receivables = ["BHCKF185", "BHCKC880"]

# Convert relevant columns to float
df_all_final[
    cols_loans_realestate
    + cols_loans_farmers
    + cols_loans_commerce_industry
    + cols_loans_households
    + cols_loans_foreign_gov
    + cols_loans_lease_financing
    + cols_loans_financial_receivables
] = df_all_final[
    cols_loans_realestate
    + cols_loans_farmers
    + cols_loans_commerce_industry
    + cols_loans_households
    + cols_loans_foreign_gov
    + cols_loans_lease_financing
    + cols_loans_financial_receivables
].astype(
    float
)

# Create aggregated columns for each category
df_all_final["CHARGED_OFF_REAL_ESTATE_FRY9C"] = df_all_final[cols_loans_realestate].sum(
    axis=1
)
df_all_final["CHARGED_OFF_FARMERS_FRY9C"] = df_all_final[cols_loans_farmers].sum(axis=1)
df_all_final["CHARGED_OFF_COMMERCE_INDUSTRY_FRY9C"] = df_all_final[
    cols_loans_commerce_industry
].sum(axis=1)
df_all_final["CHARGED_OFF_HOUSEHOLDS_FRY9C"] = df_all_final[cols_loans_households].sum(
    axis=1
)
df_all_final["CHARGED_OFF_FOREIGN_GOV_FRY9C"] = df_all_final[
    cols_loans_foreign_gov
].sum(axis=1)
df_all_final["CHARGED_OFF_LEASE_FINANCING_FRY9C"] = df_all_final[
    cols_loans_lease_financing
].sum(axis=1)

C:\Users\joaof\AppData\Local\Temp\ipykernel_37276\1731470249.py:45: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_all_final["CHARGED_OFF_REAL_ESTATE_FRY9C"] = df_all_final[cols_loans_realestate].sum(
C:\Users\joaof\AppData\Local\Temp\ipykernel_37276\1731470249.py:48: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_all_final["CHARGED_OFF_FARMERS_FRY9C"] = df_all_final[cols_loans_farmers].sum(axis=1)
C:\Users\joaof\AppData\Local\Temp\ipykernel_37276\1731470249.py:49: PerformanceWarning: DataFrame is highly fragmented.  This is

In [48]:
# 9C - recoveries:
# loans secured by real estate: BHCKC892 , BHCKC894 , BHCK3585 , BHCK5412 , BHCKC217 , BHCKC218 , BHCK3589, BHCKC896 , BHCKC898 , BHCKB513
# loans to farmers: BHCK4665
# loans to commerce and industry: BHCK4617, BHCK4618
# loans to households: BHCKB515, BHCKK133, BHCKK206
# 6 Loans to foreign governments and official institutions:	BHCK4627
# 7 Lease financing receivables:	BHCK4628
# lease to financial receivables: BHCKF187, BHCKF188

In [49]:
# Define recovery columns for each category
cols_recoveries_realestate = [
    "BHCKC892",
    "BHCKC894",
    "BHCK3585",
    "BHCK5412",
    "BHCKC217",
    "BHCKC218",
    "BHCK3589",
    "BHCKC896",
    "BHCKC898",
    "BHCKB513",
]
cols_recoveries_farmers = ["BHCK4665"]
cols_recoveries_commerce_industry = ["BHCK4617", "BHCK4618"]
cols_recoveries_households = ["BHCKB515", "BHCKK133", "BHCKK206"]
cols_recoveries_foreign_gov = ["BHCK4627"]
cols_recoveries_lease_financing = ["BHCK4628"]
cols_recoveries_financial_receivables = ["BHCKF187", "BHCKF188"]

# Convert relevant columns to float
df_all_final[
    cols_recoveries_realestate
    + cols_recoveries_farmers
    + cols_recoveries_commerce_industry
    + cols_recoveries_households
    + cols_recoveries_foreign_gov
    + cols_recoveries_lease_financing
    + cols_recoveries_financial_receivables
] = df_all_final[
    cols_recoveries_realestate
    + cols_recoveries_farmers
    + cols_recoveries_commerce_industry
    + cols_recoveries_households
    + cols_recoveries_foreign_gov
    + cols_recoveries_lease_financing
    + cols_recoveries_financial_receivables
].astype(
    float
)

# Create aggregated columns for each category
df_all_final["RECOVERIES_REAL_ESTATE_FRY9C"] = df_all_final[
    cols_recoveries_realestate
].sum(axis=1)
df_all_final["RECOVERIES_FARMERS_FRY9C"] = df_all_final[cols_recoveries_farmers].sum(
    axis=1
)
df_all_final["RECOVERIES_COMMERCE_INDUSTRY_FRY9C"] = df_all_final[
    cols_recoveries_commerce_industry
].sum(axis=1)
df_all_final["RECOVERIES_HOUSEHOLDS_FRY9C"] = df_all_final[
    cols_recoveries_households
].sum(axis=1)
df_all_final["RECOVERIES_FOREIGN_GOV_FRY9C"] = df_all_final[
    cols_recoveries_foreign_gov
].sum(axis=1)
df_all_final["RECOVERIES_LEASE_FINANCING_FRY9C"] = df_all_final[
    cols_recoveries_lease_financing
].sum(axis=1)

C:\Users\joaof\AppData\Local\Temp\ipykernel_37276\3359170036.py:43: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_all_final["RECOVERIES_REAL_ESTATE_FRY9C"] = df_all_final[
C:\Users\joaof\AppData\Local\Temp\ipykernel_37276\3359170036.py:46: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_all_final["RECOVERIES_FARMERS_FRY9C"] = df_all_final[cols_recoveries_farmers].sum(
C:\Users\joaof\AppData\Local\Temp\ipykernel_37276\3359170036.py:49: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling 

### Ratios from Consolidated reports

BHCA (ignored BHCW):
-  Common equity tier 1 capital ratio : P793
-  Tier 1 capital ratio: 7206
-  Total capital ratio: 7205
- Tier 1 leverage ratio: 7204
- Capital conservation buffer: H311

In [50]:
## add leverage ratios col - ratio common equity tier 1 leverage
cols_capital_conservation_buffer_FRY9C = ["BHCAH311"]

df_all_final[cols_capital_conservation_buffer_FRY9C] = df_all_final[
    cols_capital_conservation_buffer_FRY9C
].astype(float)

df_all_final["RATIO_CAPITAL_CONSERVATION_BUFFER_FRY9C"] = df_all_final[
    cols_capital_conservation_buffer_FRY9C
].sum(axis=1)

C:\Users\joaof\AppData\Local\Temp\ipykernel_37276\1316763027.py:8: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_all_final["RATIO_CAPITAL_CONSERVATION_BUFFER_FRY9C"] = df_all_final[


In [51]:
## add leverage ratios col - ratio common equity tier 1 leverage
cols_ratio_tier_1_leverage_FRY9C = ["BHCA7206"]

df_all_final[cols_ratio_tier_1_leverage_FRY9C] = df_all_final[
    cols_ratio_tier_1_leverage_FRY9C
].astype(float)

df_all_final["RATIO_TIER_1_LEVERAGE_FRY9C"] = df_all_final[
    cols_ratio_tier_1_leverage_FRY9C
].sum(axis=1)

C:\Users\joaof\AppData\Local\Temp\ipykernel_37276\2398034000.py:8: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_all_final["RATIO_TIER_1_LEVERAGE_FRY9C"] = df_all_final[


In [52]:
## add capital ratios col - ratio common equity tier 1 capital
cols_ratio_total_capital_FRY9C = ["BHCA7205"]

df_all_final[cols_ratio_total_capital_FRY9C] = df_all_final[
    cols_ratio_total_capital_FRY9C
].astype(float)

df_all_final["RATIO_TOTAL_CAPITAL_FRY9C"] = df_all_final[
    cols_ratio_total_capital_FRY9C
].sum(axis=1)

C:\Users\joaof\AppData\Local\Temp\ipykernel_37276\540141324.py:8: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_all_final["RATIO_TOTAL_CAPITAL_FRY9C"] = df_all_final[


In [53]:
## add capital ratios col - ratio common equity tier 1 capital
cols_ratio_tier_1_capital_FRY9C = ["BHCA7206"]

df_all_final[cols_ratio_tier_1_capital_FRY9C] = df_all_final[
    cols_ratio_tier_1_capital_FRY9C
].astype(float)

df_all_final["RATIO_TIER_1_CAPITAL_FRY9C"] = df_all_final[
    cols_ratio_tier_1_capital_FRY9C
].sum(axis=1)

C:\Users\joaof\AppData\Local\Temp\ipykernel_37276\1678517601.py:8: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_all_final["RATIO_TIER_1_CAPITAL_FRY9C"] = df_all_final[


In [54]:
## add capital ratios col - ratio common equity tier 1 capital
cols_ratio_common_equity_tier1_capital_FRY9C = ["BHCAP793"]

df_all_final[cols_ratio_common_equity_tier1_capital_FRY9C] = df_all_final[
    cols_ratio_common_equity_tier1_capital_FRY9C
].astype(float)

df_all_final["RATIO_COMMON_EQUITY_TIER1_CAPITAL_FRY9C"] = df_all_final[
    cols_ratio_common_equity_tier1_capital_FRY9C
].sum(axis=1)

C:\Users\joaof\AppData\Local\Temp\ipykernel_37276\3154768299.py:8: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_all_final["RATIO_COMMON_EQUITY_TIER1_CAPITAL_FRY9C"] = df_all_final[


## Cummulative to per quarter features



Based on the reviewed documentation, we identified that the reported income statement figures (e.g., revenue, costs, profit) represent year-to-date values for each fiscal year. This means that each quarter’s figures are cumulative for that year, resetting at the beginning of the first quarter.

In contrast, the balance sheet figures (e.g., assets, liabilities, equity) reflect the position at the end of each quarter and do not reset at the start of the year.

Therefore, for income statement data, we chose to estimate the quarterly contribution to eliminate the seasonality effect caused by the reset in the first quarter of each year.


Sources:
- FR Y -9c 
    - financial statement: https://www.federalreserve.gov/reportforms/forms/FR_Y-9C20180930_f.pdf
    - detail on financial statement rubrics: https://www.federalreserve.gov/apps/reportingforms/Download/DownloadAttachment?guid=d036ea09-75d3-4f2f-8fe3-0f43c76d2f70

- FR_Y-9LP (Parent Company Only Financial Statements for Large Bank Holding Companies): 
    - financial statement: https://www.federalreserve.gov/apps/reportingforms/Download/DownloadAttachment?guid=e56cd829-8369-4263-96ca-4de4e12ce585#:~:text=03/2024-,For%20Federal%20Reserve%20Bank%20Use%20Only,for%20investments%20in%20equity%20securities.
    - detail on financial statement rubrics: https://www.federalreserve.gov/reportforms/forms/FR_Y-9LP20220705_i.pdf 

- FR Y-9SP (Parent Company Only Financial Statements for Small Holding Companies) - only reports data semi-annually 
    - financial statement: https://www.federalreserve.gov/apps/reportingforms/Download/DownloadAttachment?guid=9df5bb23-468c-4407-a874-7beaba03b930
    - detail on financial statement rubrics: https://www.federalreserve.gov/apps/reportingforms/Download/DownloadAttachment?guid=05505b4f-780d-49e0-a123-59e7cbf29afb


In [57]:
df_all_final_cols = pd.Series(df_all_final.columns)

In [58]:
list_cols_cummulative = df_all_final_cols[
    df_all_final_cols.str.startswith(
        ("REVENUE", "NET_INCOME", "CHARGED", "RECOVERIES", "LOANS", "REVENUE", "EBT")
    )
].unique()

In [59]:
def calc_per_quarter_value(list_report_col, df):
    """
    Convert cumulative reporting columns into per-quarter values while
    handling late-starting companies (first reported quarter > baseline).
    """

    df = df.sort_values(["id", "year", "nr_quarter"]).copy()

    for report_col in list_report_col:

        # determine baseline
        if "FRY9SP" in report_col:
            quarter_baseline = 2
        elif "FRY9LP" in report_col or "FRY9C" in report_col:
            quarter_baseline = 1
        else:
            quarter_baseline = 1

        cum_col = f"{report_col}_CUMMULATIVE"
        df[cum_col] = df[report_col]

        # main diff per id + year
        diff_vals = df.groupby(["id", "year"])[report_col].diff()
        df[report_col] = diff_vals

        # rule 1: if nr_quarter <= baseline → keep cumulative
        baseline_mask = df["nr_quarter"] <= quarter_baseline
        df.loc[baseline_mask, report_col] = df.loc[baseline_mask, cum_col]

        # rule 2: if company first reports AFTER baseline → keep cumulative at that first report
        # first observed quarter within (id, year)
        first_quarter = df.groupby(["id", "year"])["nr_quarter"].transform("min")
        first_obs_mask = (df["nr_quarter"] == first_quarter) & (
            df["nr_quarter"] > quarter_baseline
        )
        df.loc[first_obs_mask, report_col] = df.loc[first_obs_mask, cum_col]

    return df

In [63]:
df_all_final.head()

,BHCA7204,BHCA7205,BHCA7206,BHCAH311,BHCAP793,BHCK0081,BHCK0395,BHCK0397,BHCK1292,BHCK1296,...,RECOVERIES_FARMERS_FRY9C,RECOVERIES_COMMERCE_INDUSTRY_FRY9C,RECOVERIES_HOUSEHOLDS_FRY9C,RECOVERIES_FOREIGN_GOV_FRY9C,RECOVERIES_LEASE_FINANCING_FRY9C,RATIO_CAPITAL_CONSERVATION_BUFFER_FRY9C,RATIO_TIER_1_LEVERAGE_FRY9C,RATIO_TOTAL_CAPITAL_FRY9C,RATIO_TIER_1_CAPITAL_FRY9C,RATIO_COMMON_EQUITY_TIER1_CAPITAL_FRY9C
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [89]:
# Mapping dictionary: date string -> quarter code
quarter_map = {
    "1231": " Q4",
    "0331": " Q1",
    "0630": " Q2",
    "0930": " Q3"  
}

# Replace using .replace with a dictionary
df_all_final['quarter_code'] = df_all_final['quarter'].replace(quarter_map, regex=True)

In [90]:
df_all_final["year"] = df_all_final["quarter"].str[:4].astype(int)
df_all_final["month"] = df_all_final["quarter"].str[4:6].astype(int)
df_all_final["nr_quarter"] = ((df_all_final["month"] - 1) // 3) + 1

structure_cols = {"RSSD9017": "legal_name", "RSSD9001": "id", "RSSD9005": "country"}
df_all_final_rename = df_all_final.rename(columns=structure_cols)


In [91]:
df_all_final_cols_wcummulative = calc_per_quarter_value(list_report_col=list_cols_cummulative, df=df_all_final_rename)
df_all_final_cols_wcummulative.head()

,BHCA7204,BHCA7205,BHCA7206,BHCAH311,BHCAP793,BHCK0081,BHCK0395,BHCK0397,BHCK1292,BHCK1296,...,CHARGED_OFF_COMMERCE_INDUSTRY_FRY9C_CUMMULATIVE,CHARGED_OFF_HOUSEHOLDS_FRY9C_CUMMULATIVE,CHARGED_OFF_FOREIGN_GOV_FRY9C_CUMMULATIVE,CHARGED_OFF_LEASE_FINANCING_FRY9C_CUMMULATIVE,RECOVERIES_REAL_ESTATE_FRY9C_CUMMULATIVE,RECOVERIES_FARMERS_FRY9C_CUMMULATIVE,RECOVERIES_COMMERCE_INDUSTRY_FRY9C_CUMMULATIVE,RECOVERIES_HOUSEHOLDS_FRY9C_CUMMULATIVE,RECOVERIES_FOREIGN_GOV_FRY9C_CUMMULATIVE,RECOVERIES_LEASE_FINANCING_FRY9C_CUMMULATIVE
314775,NaN,NaN,NaN,NaN,NaN,130288,1359,0,0,0,...,269.0,0.0,0.0,0.0,1.0,78.0,62.0,0.0,0.0,0.0
308921,NaN,NaN,NaN,NaN,NaN,137778,1887,0,0,0,...,583.0,0.0,0.0,0.0,6.0,130.0,167.0,0.0,0.0,0.0
306773,NaN,NaN,NaN,NaN,NaN,127964,1915,0,0,0,...,2661.0,0.0,0.0,0.0,5.0,144.0,257.0,0.0,0.0,0.0
300917,NaN,NaN,NaN,NaN,NaN,194450,1831,0,0,0,...,3273.0,0.0,0.0,0.0,7.0,168.0,320.0,0.0,0.0,0.0
298721,NaN,NaN,NaN,NaN,NaN,133948,769,0,0,0,...,2142.0,82.0,0.0,0.0,9.0,47.0,164.0,9.0,0.0,0.0


# Save data in output

In [92]:
cols = [c for c in df_all_final_cols_wcummulative.columns if c[:2] != "BH" and c[:2] != "RS"]
df_filtered = df_all_final_cols_wcummulative.loc[:, cols]


In [94]:
main_dir = "../outputs/all_banks/"

if os.path.exists(main_dir) == False:
    os.makedirs(main_dir, exist_ok=False)

chunk_size = 50_000
for i in range(0, len(df_filtered), chunk_size):
    chunk = df_filtered.iloc[i : i + chunk_size]
    chunk.to_pickle(f"{main_dir}df_banks_part_{i//chunk_size + 1}.pickle")

In [87]:
df_report_assets.to_pickle("../outputs/df_bis.pickle")